# 拓展：ToT 与工具依赖调度

本页学习目标：

- 理解保留多个候选方案的用途与成本。
- 区分候选搜索树和工具任务依赖图。
- 判断哪些工具可以并行，哪些必须等待。
- 处理搜索预算不足、依赖错误和工具失败。

本页用小型程序说明两种不同思路，没有复现相关论文的完整系统或成绩。

[返回课程入口](../README.md) · 理解基本循环后再看。


## 1. ToT：先保留多个候选

只保留当前评分最高的选择，可能很快走进死路。本页虚构搜索树中，候选 A 的初始分数比 B 高，但 A 没有可完成路径，B 可以到达目标。

- 每层只保留一个候选时，选择 A，最终进入死路。
- 每层保留两个候选时，可以继续探索 B 并到达目标。

这说明多个候选能避免某些过早取舍，也会增加展开次数。分数只是启发式判断；评分错误、候选缺失或预算不足仍然会失败。


## 2. 用算术例子理解搜索

本页还用 2、3、4 按顺序选择加法或乘法，目标为 20。

正确路径之一是 `(2 + 3) * 4 = 20`。如果第一步只看“离20更近”，可能先选 `2 * 3 = 6`，提前丢掉能够到达答案的 `2 + 3 = 5`。

程序要记录生成了哪些候选、保留和删掉哪些，以及为什么停止。这里的默认候选由规则产生，不是模型搜索成绩。


## 3. 工具依赖图解决什么问题

假设要买三件商品，需要查询单价、查询库存，然后汇总总价与是否有货。

```text
查询单价 ──┐
           ├── 汇总
查询库存 ──┘
```

单价和库存互不依赖，可以并行查询；汇总必须等待两者成功。依赖图的节点是需要执行的任务，不是竞争中的候选答案。

本页教学数据是单价 3、库存 10，最终应得到总价 9、库存足够。这些是固定课堂值。


## 4. 调度前后要检查什么

执行前检查任务 ID 唯一、工具存在、依赖存在、依赖图没有环。执行中只启动依赖已满足的任务，保留各任务的输入、输出、开始和结束时间。

一个前置任务失败后，不能继续运行依赖其结果的汇总。本页模拟单价查询失败时，检查汇总没有执行。

并行能减少相互独立操作的等待，但不自动提高模型判断质量。共享机器的一次计时也不能证明通用加速比例，因此代码检查结果和依赖顺序，不强求固定加速倍数。


## 5. 什么时候需要这些方法

| 情况 | 可以考虑什么 |
| --- | --- |
| 过早选定一个解释容易出错 | 保留多个候选，再用证据检查 |
| 多个独立查询主要耗在等待 | 按依赖关系并行 |
| 一步需要上一工具返回的 ID | 顺序执行，显式传递结果 |
| 简单读取已经够用 | 保持简单循环 |

模型调用、候选展开、工具执行和总时间需要共同计数。新开分支或子任务，不应把整个任务的预算重新归零。


## 6. 练习与代码说明

把树宽从 2 改为 1，观察保留路径的变化。再给汇总加一个不存在的依赖，检查是否在执行前被拒绝。

`tree_search` 检查候选搜索，`validate_dag` 检查依赖图，`schedule` 执行工具，`arithmetic_search` 演示算术候选。默认策略和数据由程序定义；最后的真实模型分支默认关闭。

下一章：[多 Agent 协作](../04-multi-agent/01-coordination.ipynb)。


In [ ]:
# 虚构的方案搜索：看起来更好的 A 会走向死路，B 起初评分低但可完成。
children = {"start": ["A", "B"], "A": ["A_dead"], "B": ["B_goal"],
            "A_dead": [], "B_goal": []}
value = {"start": 0, "A": 9, "B": 6, "A_dead": -1, "B_goal": 10}
def propose(node):
    return children[node]
def score(node):
    return value[node]
def is_goal(node):
    return node == "B_goal"

print("第一步候选：", [(node, score(node)) for node in propose("start")])

In [ ]:
def tree_search(width=1, max_depth=3, max_expansions=8):
    if width < 1 or max_depth < 0 or max_expansions < 0:
        raise ValueError("非法搜索预算")
    frontier, trace, expanded = [("start",)], [], 0
    for depth in range(max_depth):
        candidates = []
        for path in frontier:
            if expanded >= max_expansions:
                return {"status": "expansion_limit", "trace": trace, "expanded": expanded}
            expanded += 1
            for node in propose(path[-1]):
                if node not in path:  # 防止沿当前路径循环
                    candidates.append(path + (node,))
        ranked = sorted(candidates, key=lambda path: score(path[-1]), reverse=True)
        kept = ranked[:width]
        trace.append({"depth": depth+1, "candidates": candidates, "kept": kept,
                      "pruned": ranked[width:]})
        for path in kept:
            if is_goal(path[-1]):
                return {"status": "solved", "path": path, "trace": trace, "expanded": expanded}
        frontier = [path for path in kept if children[path[-1]]]
        if not frontier:
            return {"status": "dead_end", "trace": trace, "expanded": expanded}
    return {"status": "depth_limit", "trace": trace, "expanded": expanded}

greedy, beam = tree_search(width=1), tree_search(width=2)
print("贪心：", greedy)
print("保留两个：", beam)
assert greedy["status"] == "dead_end"
assert beam["status"] == "solved" and beam["path"] == ("start", "B", "B_goal")
assert beam["expanded"] > greedy["expanded"]

In [ ]:
assert tree_search(width=2, max_expansions=1)["status"] == "expansion_limit"
assert tree_search(width=2, max_depth=1)["status"] == "depth_limit"
print("预算限制先于理想结果；增大树宽不保证在给定预算下成功。")

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy

tasks = [
    {"id": "price", "tool": "price", "deps": []},
    {"id": "stock", "tool": "stock", "deps": []},
    {"id": "summary", "tool": "summary", "deps": ["price", "stock"]},
]

def validate_dag(plan):
    if not isinstance(plan, list) or not plan or len(plan) > 20:
        raise ValueError("计划须包含 1–20 个任务")
    if any(not isinstance(task, dict) or set(task) != {"id", "tool", "deps"} for task in plan):
        raise ValueError("任务字段应为 id/tool/deps")
    ids = [task["id"] for task in plan]
    if any(not isinstance(i, str) or not i for i in ids) or len(set(ids)) != len(ids):
        raise ValueError("任务 ID 应为非空且唯一")
    for task in plan:
        deps = task["deps"]
        if not isinstance(deps, list) or any(not isinstance(d, str) or d not in ids for d in deps) or len(set(deps)) != len(deps):
            raise ValueError("缺失或重复依赖")
        if not isinstance(task["tool"], str) or task["tool"] not in {"price", "stock", "summary"}:
            raise ValueError("未知工具")
    # 拓扑检查：反复移走没有剩余前置的节点，移不动说明有环。
    pending, done = list(plan), set()
    while pending:
        ready = [task for task in pending if set(task["deps"]) <= done]
        if not ready:
            raise ValueError("依赖图存在环")
        done.update(task["id"] for task in ready)
        pending = [task for task in pending if task["id"] not in done]
    return True

assert validate_dag(tasks)
for bad in [tasks + [tasks[0]], [{"id": "missing_fields"}],
            [{"id": "x", "tool": "price", "deps": ["missing"]}],
            [{"id": "x", "tool": "price", "deps": ["x"]}]]:
    try:
        validate_dag(bad)
    except ValueError as error:
        print("已拒绝：", error)
    else:
        raise AssertionError("非法计划不应通过")

In [ ]:
def execute_task(task, inputs, fail=None):
    started = time.perf_counter()
    try:
        if task["id"] == fail:
            raise RuntimeError("模拟资料服务失败")
        if task["tool"] == "price":
            time.sleep(0.06)  # 本地可控的等待，不访问外部网络
            output = 3
        elif task["tool"] == "stock":
            time.sleep(0.06)
            output = 10
        else:
            output = {"total": inputs["price"] * 3, "available": inputs["stock"] >= 3}
        return {"id": task["id"], "ok": True, "output": output,
                "start": started, "end": time.perf_counter()}
    except Exception as error:
        return {"id": task["id"], "ok": False, "error": str(error),
                "start": started, "end": time.perf_counter()}

def schedule(plan, parallel=True, fail=None):
    validate_dag(plan)
    pending, results, trace = list(plan), {}, []
    begin = time.perf_counter()
    with ThreadPoolExecutor(max_workers=2 if parallel else 1) as pool:
        while pending:
            ready = [task for task in pending if set(task["deps"]) <= set(results)]
            if not ready:
                return {"status": "blocked", "trace": trace}
            futures = [pool.submit(execute_task, task,
                                   {d: deepcopy(results[d]) for d in task["deps"]}, fail)
                       for task in ready]
            batch = [future.result() for future in futures]
            trace.extend(batch)
            if any(not event["ok"] for event in batch):
                return {"status": "tool_error", "trace": trace, "results": results}
            results.update({event["id"]: event["output"] for event in batch})
            pending = [task for task in pending if task["id"] not in results]
    return {"status": "completed", "results": results, "trace": trace,
            "latency_ms": (time.perf_counter()-begin)*1000}

serial, parallel = schedule(tasks, parallel=False), schedule(tasks, parallel=True)
assert serial["results"] == parallel["results"]
assert parallel["results"]["summary"] == {"total": 9, "available": True}
records = {e["id"]: e for e in parallel["trace"]}
assert records["summary"]["start"] >= max(records["price"]["end"], records["stock"]["end"])
overlap = max(records["price"]["start"], records["stock"]["start"]) < min(records["price"]["end"], records["stock"]["end"])
print({"serial_ms": round(serial["latency_ms"], 2), "parallel_ms": round(parallel["latency_ms"], 2),
       "observed_overlap": overlap, "same_result": serial["results"] == parallel["results"]})
# 不断言耗时比例：机器负载会影响测量，单次结果不能推出通用加速比。

In [ ]:
failed = schedule(tasks, fail="price")
assert failed["status"] == "tool_error"
assert not any(event["id"] == "summary" for event in failed["trace"])
print(failed)

In [ ]:
def arithmetic_search(policy, width=2, max_calls=3):
    if width < 1 or max_calls < 0:
        raise ValueError("非法预算")
    frontier = [{"value": 2, "expression": "2", "remaining": [3, 4]}]
    trace, used = [], 0
    for depth in range(2):
        candidates = []
        for state in frontier:
            if used >= max_calls:
                return {"status": "call_limit", "trace": trace}
            used += 1
            proposal = policy(deepcopy(state))
            choices = proposal.get("choices") if isinstance(proposal, dict) else None
            if not isinstance(choices, list) or not 1 <= len(choices) <= 2:
                return {"status": "invalid_proposal", "trace": trace}
            seen_ops = set()
            for choice in choices:
                if (not isinstance(choice, dict) or not isinstance(choice.get("op"), str) or choice["op"] not in {"+", "*"} or
                    type(choice.get("score")) not in (int, float) or not 0 <= choice["score"] <= 1 or
                    choice["op"] in seen_ops):
                    return {"status": "invalid_proposal", "trace": trace}
                op = choice["op"]
                seen_ops.add(op)
                next_number = state["remaining"][0]
                value = state["value"] + next_number if op == "+" else state["value"] * next_number
                candidates.append({"value": value, "expression": f"({state['expression']}{op}{next_number})",
                                   "remaining": state["remaining"][1:], "score": choice["score"]})
        frontier = sorted(candidates, key=lambda row: row["score"], reverse=True)[:width]
        trace.append({"depth": depth+1, "candidates": candidates, "kept": deepcopy(frontier)})
    solutions = [state for state in frontier if state["value"] == 20]
    return {"status": "solved" if solutions else "not_solved", "solutions": solutions, "trace": trace, "calls": used}

def candidate_fixture(state):
    # 两种操作都生成，用候选值距目标的距离作局部评分；这个启发式会误导贪心。
    n = state["remaining"][0]
    return {"choices": [{"op": "+", "score": 1/(1+abs(20-(state["value"]+n)))},
                        {"op": "*", "score": 1/(1+abs(20-(state["value"]*n)))}]}

assert arithmetic_search(candidate_fixture, width=1)["status"] == "not_solved"
arithmetic = arithmetic_search(candidate_fixture, width=2)
assert arithmetic["status"] == "solved" and arithmetic["solutions"][0]["expression"] == "((2+3)*4)"
assert arithmetic_search(candidate_fixture, max_calls=1)["status"] == "call_limit"
assert arithmetic_search(lambda state: {"choices": [{"op": "exec", "score": 1}]})["status"] == "invalid_proposal"
print(arithmetic)

In [ ]:
import json
from urllib.request import Request, build_opener, ProxyHandler

def ollama_json(instruction, state, model="qwen3:1.7b"):
    # 单次 JSON 决策接口；没有内置 agent loop。程序仍须校验返回字段。
    request_data = {
        "model": model, "stream": False, "format": "json",
        "messages": [{"role": "system", "content": instruction},
                     {"role": "user", "content": json.dumps(state, ensure_ascii=False)}],
    }
    request = Request("http://127.0.0.1:11434/api/chat",
                      json.dumps(request_data).encode("utf-8"), {"Content-Type": "application/json"})
    with build_opener(ProxyHandler({})).open(request, timeout=60) as response:
        payload = json.load(response)
    return json.loads(payload["message"]["content"])

In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    def live_candidates(state):
        return ollama_json(
            '目标是从value开始，依次使用remaining的每个数，选+或*最终得到20。'
            '返回1到2个候选操作及到达目标的前景评分：'
            '{"choices":[{"op":"+或*","score":0到1之间数字}]}。不要输出任意代码。', state)
    live_tree = arithmetic_search(live_candidates)
    print("真实模型候选搜索：", live_tree)
    assert live_tree["status"] == "solved", "未解出也是有效实验结果；检查候选遗漏与评分剪枝"

    plan = ollama_json(
        '任务：查询单价和库存后汇总买三件的总价与可用性。只输出 {"tasks":[...]}。'
        '节点固定为 price、stock、summary，每个节点含 id、同名tool、deps列表。'
        'price与stock独立；summary执行时必须已得到price与stock。',
        {"tools": ["price", "stock", "summary"]})
    live_schedule = schedule(plan["tasks"])
    print("模型规划、程序执行：", live_schedule)
    assert live_schedule["status"] == "completed"
    assert live_schedule["results"]["summary"] == {"total": 9, "available": True}
else:
    print("真实模型 ToT/DAG 规划未运行；已执行的是本地控制流实验。")